# Introducing `pathpyG`

*August 4 2026*  
*Training Workshop: Introduction to Deep Graph Learning*  
*Ingo Scholtes, CAIDAS, Julius-Maximilians-Universität Würzburg (JMU), Germany*  


We first introduce `pathpyG`, a graph learning and visualization package that is being developed at my chair at University of Wuerzburg.

`pathpyG` has a couple of advantages that motivate the use across the two training workshops. First, it is easy to install since it is a pure `python` package that does not require compilation. Second, `pathpyG` has a user-friendly API that makes it easy to handle directed and undirected networks, networks where nodes or edges have attributes as well as temporal networks. Third, it provides interactive HTML visualizations that can be directly displayed inside `jupyter` notebooks, making it particularly suitable for educational settings. Moreover, it directly supports the analysis and visualization of time series data on networked systems, such as time-stamped edges or data on paths in networks.

Finally and most importantly, `pathpyG` is based on `pyTorch` and `torch-geometric` and uses tensors as internal data representation. This facilitates the analysis of large data sets on the GPU and makes it easy to apply graph neural networks in later chapters of our tutorials.

To get started, we first import `pathpyG` and assign the local alias `pp`:

In [ ]:
import pathpyG as pp

## Creating networks

`pathpyG` provides the `Graph` class. The constructor takes a `pyG` Data object that can be used to pass an edge-index that captures the edges of a graph, as well as arbitrary node-, edge- or graph-level attributes. To simplify the creation of small example networks, you can use a static function that create a `Graph` object based on a list of edges represented as tuples of integers or strings.

Printing the `Graph` object will give a short string summary which tells whether the network is directed or undirected, as well as the number of unique nodes and links.

In [3]:
g1 = pp.Graph.from_edge_list([(0, 1), (1, 2), (2, 0)])
print(g1)

Directed graph with 3 nodes and 3 edges
{'Edge Attributes': {}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}


A network is directed by default, but we can create an undirected network by calling the function `to_undirected` of the `Graph` instance. This will internally generate edges for all directions, i.e. for the example above it will additionally generate the edges that connect nodes in the opposite direction.

In [4]:
g2 = g1.to_undirected()
print(g2)

Undirected graph with 3 nodes and 6 (directed) edges
{'Edge Attributes': {}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}


In the example above, we have used integer numbers to refer to different nodes. Internally, this will create a `pyG` Data object with an edge index, where nodes are always represented by integer indices. Let us have a look at this internal data structure:

In [5]:
print(g2.data.edge_index)

EdgeIndex([[0, 0, 1, 1, 2, 2],
           [1, 2, 0, 2, 0, 1]], sparse_size=(3, 3), nnz=6, sort_order=row,
          is_undirected=True)


We see that this edge index contains all edges for both directions, where the first tensor contains all source nodes (ordered by their index) while the second tensor contains all target nodes. This sorted tensor representation allows us to easy convert the edge index tensor to sparse matrix representations, that can be used e.g. to calculate matrix-based measures or Laplacian operators.

While the approach to use integers is easy to understand, it is often more convenient to use string-based node labels. Different from `pyG`, this is supported in `pathpyG`, i.e. we can create a network as follows:

In [6]:
n = pp.Graph.from_edge_list([('Tom', 'Bert'), ('Bert', 'Bill'), ('Bill', 'Tom')]).to_undirected()
print(n)

Undirected graph with 3 nodes and 6 (directed) edges
{'Edge Attributes': {}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}


Checking the edge index, we find that we again have integer-based node indices:

In [7]:
n.data.edge_index

EdgeIndex([[0, 0, 1, 1, 2, 2],
           [1, 2, 0, 2, 0, 1]], sparse_size=(3, 3), nnz=6, sort_order=row,
          is_undirected=True)

However, `pathpyG` automatically generates a ID to index mapping object that is automatically applied when we e.g. enumerate through nodes. We can manually check this mapping and use it to resolve IDs to integer indices and vice-versa:

In [8]:
print(n.mapping)

Bert -> 0
Bill -> 1
Tom -> 2



In [9]:
n.mapping.to_idx('Tom')

2

In [10]:
n.mapping.to_idx('Bert')

0

If we want to check explicitly whether a node exists before creating and edge, we can test this with the `in` operator on the set of nodes available via `Graph.nodes`:

In [11]:
print('Tom' in n.nodes)

True


To count the number of nodes and (directed) edges in a network we can use the `n` and `m` attributes:

In [12]:
print('Network has {0} nodes and {1} edges'.format(n.n, n.m))

Network has 3 nodes and 6 edges


### Enumerating nodes and edges

We can iterate through nodes via the nodes iterator as follows. If the Graph object includes an index-ID mapping, this will be applied automatically:

In [13]:
for v in n.nodes:
    print(v)

Bert
Bill
Tom


Similar to `nodes`, the `edges` iterator of the network contains all edges of a network. Each edge is returned as a tuple and the id-index mapping is applied automatically if such a mapping exists (otherwise we obtain a tuple of node indices):

In [14]:
for e in n.edges:
    print(e)

('Bert', 'Bill')
('Bert', 'Tom')
('Bill', 'Bert')
('Bill', 'Tom')
('Tom', 'Bert')
('Tom', 'Bill')


We often want to check whether an edge exists between a specific pair of nodes. We can do this by using the `is_edge` function:

In [15]:
print(n.is_edge('Tom', 'Bert'))

True


We can access the degrees of nodes, i.e. the number of other nodes to which a node is connected, via the `degrees()` function of the Network. For an undirected network, the degrees() function gives the undirected degrees (i.e. irrespective of the directionality of an edge). For directed networks we can use the mode parameter to calculate the in- or out-degree of of ndoes in a directed network (i.e. to how many other nodes the edges of a node point of from how many other nodes edges point to the given node).

All of those functions return a dictionary that can be indexed via the node ids.

In [16]:
n.degrees(mode='in')['Tom']

2

In [17]:
n.degrees(mode='in')['Tom']

2

## Networks, Nodes and Edges with attributes

We often want to use networks to model relational data that contain additional information on nodes, edges, or networks. To support this, `pathpyG` stores data in terms of a pyG data frame, which allows to store arbitrary additional information at the level of nodes, edges or the graph in terms of torch.tensors.

In [18]:
n = pp.Graph.from_edge_list([('Tom', 'Bert'), ('Bert', 'Bill'), ('Bill', 'Tom')])
print(n)

Directed graph with 3 nodes and 3 edges
{'Edge Attributes': {}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}


In the following example, we add an attribute to the modes of the graph. We can directly assing this to the underylying `pyG` data object. All node attributes must be prefixed with `node_`:

In [19]:
import torch

n.data.node_age = torch.tensor([[44], [28], [125]])

The assignment of these values to the nodes is based on the indices of nodes, which we can check via the mapping object. We can now use the following code to access the properties of individual nodes:

In [20]:
n['node_age', 'Bill']

tensor([28])

To retrieve the tensor containing the ages of all nodes, we can do the following:

In [21]:
n['node_age']

tensor([[ 44],
        [ 28],
        [125]])

Just like nodes, `Edge` objects can store arbitrary attributes that we can add as a tensor. The name of the attribute must be prefixed by `edge_`

In [22]:
n.data.edge_type = torch.tensor([[1], [2], [1]])

We can access those as follows:

In [23]:
print(n['edge_type'])
print(n['edge_type', 'Tom', 'Bert'])

tensor([[1],
        [2],
        [1]])
tensor([1])


## Adjacency and Laplacian matrices

Adjacency matrices and Laplacian matrices are important mathematical representations of networks. 

### Adjacency matrix

The topology of a graph can be represented in the entries of a matrix $A$, where an entry $A[i,j]=1$ indicates that an edge exists from the i-th to the j-th node of the network. The absence of edges is encoded by zero entries. The size of an adjacency matrix representation of a network with n nodes is generally $n^2$, which is not suitable for networks with thousands or millions of nodes. `pathpyG` nevertheless supports efficient adjacency matrix calculation for *sparse* networks, i.e. networks where the majority of node pairs are not connected by an edge. Instead of a fully populated matrix, a call to `Graph.sparse_adj_matrix()` returns a *sparse matrix object*, which is an efficient adjacency-list representation capturing the indices and values of non-zero entries.

### Laplacian matrix

The **Laplacian matrix** is a second, closely related representation that plays a central role in spectral graph theory. For an undirected graph with adjacency matrix $A$ and diagonal degree matrix $D$ (where $D_{ii}$ is the degree of the $i$-th node), the (unnormalized) Laplacian is defined as

$$ L = D - A $$

The eigenvalues and eigenvectors of $L$ reveal important structural properties of a graph. For instance, the multiplicity of the eigenvalue zero corresponds to the number of connected components of the graph, and the eigenvector belonging to the second-smallest eigenvalue (the so-called *Fiedler vector*) can be used to detect natural cluster structure in a graph. We will use exactly this idea to construct a simple, unsupervised node embedding technique called *Laplacian eigenmaps* in a later notebook of this tutorial. Just like for the adjacency matrix, `pathpyG` provides a convenience function `Graph.laplacian()` that directly returns a sparse matrix representation of the Laplacian.

In [24]:
print(n.sparse_adj_matrix())

<COOrdinate sparse matrix of dtype 'float32'
	with 3 stored elements and shape (3, 3)>
  Coords	Values
  (0, 1)	1.0
  (1, 2)	1.0
  (2, 0)	1.0


This enables us to directly apply matrix algebra operations from the sparse linear algebra module that is contained in `scipy`. If we instead want a dense matrix that includes zero entries, we can write:

In [25]:
print(n.sparse_adj_matrix().todense())

[[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]


The fact that the matrix is assymetric tells us that this is a directed network. By default, a binary matrix representation is returned where entries store the presence or absence of edges as 0 or 1 entries. If we want to use numerical attributes of edges instead, we can pass the name of a numerical attribute that should be used:

In [26]:
print(n.sparse_adj_matrix(edge_attr='edge_type').todense())

[[0 1 0]
 [0 0 2]
 [1 0 0]]


How does `pathpyG` populate adjaecency matrices if the network contains multiple edges between the same pair of nodes? Let's try this by creating another edge between Tom and Bert, and let's further add a strength attribute:

In [27]:
n = pp.Graph.from_edge_list([('Tom', 'Bert'), ('Bert', 'Bill'), ('Bill', 'Tom'), ('Tom', 'Bert')])
n.data.edge_weight=torch.tensor([[2],[0.5],[1.2],[3.7]])
print(n)

Directed graph with 3 nodes and 4 edges
{'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([4, 1])"}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}


If we now generate an adjacency matrix, the entries contain the *number of different edge objects* between pairs of nodes:

In [30]:
n.sparse_adj_matrix().todense()

matrix([[0., 1., 0.],
        [0., 0., 1.],
        [2., 0., 0.]], dtype=float32)

If we use a numerical attribute to calculate the matrix entries in such a network, the attributes of all edges between the same pair of nodes is automatically summed:

In [31]:
n.sparse_adj_matrix(edge_attr='edge_weight').todense()

matrix([[0. , 2. , 0. ],
        [0. , 0. , 0.5],
        [4.9, 0. , 0. ]], dtype=float32)

Let's compute the Laplacian matrix for the undirected triangle graph `g2` that we created at the very beginning of this notebook:

In [ ]:
print(g2.laplacian().todense())

Since `g2` is the undirected triangle graph on three nodes (each node has degree two), the diagonal entries of $L$ are all equal to two, while the off-diagonal entries are $-1$ for every pair of adjacent nodes (and would be zero for non-adjacent nodes). We can double check this by manually computing $D - A$ from the adjacency matrix and the node degrees:

In [ ]:
import scipy as sp

A = g2.sparse_adj_matrix()
D = sp.sparse.diags(g2.degrees(mode='in', return_tensor=True).numpy())
L = D - A
print(L.todense())

In many applications it is useful to work with a **normalized** Laplacian instead, since the unnormalized Laplacian is dominated by nodes with high degree. `pathpyG` supports two common normalization schemes, which can be selected via the `normalization` argument of `Graph.laplacian()`:

- `normalization='sym'` gives the **symmetric normalized Laplacian** $L_{sym} = I - D^{-1/2} A D^{-1/2}$
- `normalization='rw'` gives the **random-walk normalized Laplacian** $L_{rw} = I - D^{-1} A$, which is closely related to the transition matrix of a random walk on the graph

Let's compute the symmetric normalized Laplacian of our example graph:

In [ ]:
print(g2.laplacian(normalization='sym').todense())